# IPL Player Salary Prediction (Clean Pipeline)
End-to-end workflow: EDA, preprocessing, feature selection, models, tuning, and final deployment artifact.

## 1. Import libraries

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

In [ ]:
# Modeling and preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression

In [ ]:
# Models and metrics
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from joblib import dump

## 2. Load data

In [ ]:
# Load dataset (same source as original work)
url = 'https://raw.githubusercontent.com/devpatel0005/IPL-Player-Salary-Prediction/refs/heads/main/Player%20-%2023AIML047%20PATEL%20DEV%20DHARMESH%20-%20Player.csv'
df = pd.read_csv(url)
df.shape

In [ ]:
# Quick peek at data
df.head()

## 3. Data overview and spread

In [ ]:
# Structure and types
df.info()

In [ ]:
# Summary statistics for numeric columns
df.describe().T

In [ ]:
# Missing values check (top 20)
df.isna().sum().sort_values(ascending=False).head(20)

## 4. Define target and feature types

In [ ]:
# Set target column (change here if needed)
target_col = 'Salary'
X = df.drop(columns=[target_col])
y = df[target_col]
X.shape, y.shape

In [ ]:
# Separate numeric and categorical features
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()
print('Numeric:', numeric_cols)
print('Categorical:', cat_cols)

## 5. EDA - Univariate numeric

In [ ]:
# Histograms for first few numeric features
plt.figure(figsize=(12, 8))
for i, col in enumerate(numeric_cols[:6], 1):
    plt.subplot(2, 3, i)
    sns.histplot(df[col], kde=True)
    plt.title(col)
plt.tight_layout()
plt.show()

### EDA - Univariate categorical

In [ ]:
# Bar plots for first few categorical features
for col in cat_cols[:4]:
    vc = df[col].value_counts().head(10)
    plt.figure(figsize=(6, 3))
    sns.barplot(x=vc.index, y=vc.values)
    plt.title(col)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

### EDA - Bivariate numeric vs target

In [ ]:
# Scatter plots of numeric features vs target
for col in numeric_cols[:4]:
    plt.figure(figsize=(5, 3))
    sns.scatterplot(x=df[col], y=df[target_col])
    plt.title(f'{col} vs {target_col}')
    plt.tight_layout()
    plt.show()

### EDA - Bivariate categorical vs target

In [ ]:
# Boxplots of target across categorical levels
for col in cat_cols[:3]:
    plt.figure(figsize=(6, 3))
    sns.boxplot(x=df[col], y=df[target_col])
    plt.title(f'{target_col} by {col}')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Insights (fill after reviewing plots)
- Note key relationships and distributions here.

## 7. Preprocessing, feature selection, feature engineering

In [ ]:
# Numeric and categorical transformers
numeric_transformer = Pipeline([('scaler', StandardScaler())])
cat_transformer = Pipeline([('encoder', OneHotEncoder(handle_unknown='ignore'))])

In [ ]:
# ColumnTransformer combining numeric and categorical parts
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', cat_transformer, cat_cols)
])

In [ ]:
# Simple feature selector (can be tuned later)
feature_selector = SelectKBest(score_func=f_regression, k='all')

In [ ]:
# Placeholder for manual feature engineering (currently none)
X_fe = X.copy()
X_fe.head()

## 8. Train-test split

In [ ]:
# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_fe, y, test_size=0.2, random_state=42
)
X_train.shape, X_test.shape

## 9. Define model pipelines (with encoding pipeline)

In [ ]:
# Linear baseline
models = {}
models['Linear'] = Pipeline([
    ('pre', preprocessor),
    ('fs', feature_selector),
    ('model', LinearRegression())
])

In [ ]:
# Ridge and Lasso
models['Ridge'] = Pipeline([
    ('pre', preprocessor),
    ('fs', feature_selector),
    ('model', Ridge())
])
models['Lasso'] = Pipeline([
    ('pre', preprocessor),
    ('fs', feature_selector),
    ('model', Lasso())
])

In [ ]:
# Tree ensembles
models['RandomForest'] = Pipeline([
    ('pre', preprocessor),
    ('fs', feature_selector),
    ('model', RandomForestRegressor(random_state=42))
])
models['GradientBoosting'] = Pipeline([
    ('pre', preprocessor),
    ('fs', feature_selector),
    ('model', GradientBoostingRegressor(random_state=42))
])

## 10. Train base models and evaluate

In [ ]:
# Fit each model and collect metrics
results = []
fitted_models = {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    y_tr = pipe.predict(X_train)
    y_te = pipe.predict(X_test)
    results.append({
        'Model': name,
        'Train R2': r2_score(y_train, y_tr),
        'Test R2': r2_score(y_test, y_te),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, y_te)),
        'Test MAE': mean_absolute_error(y_test, y_te)
    })
    fitted_models[name] = pipe

In [ ]:
# Results table for base models
results_df = pd.DataFrame(results).sort_values('Test R2', ascending=False)
results_df

In [ ]:
# Bar plot of Test R2 for base models
plt.figure(figsize=(6, 4))
sns.barplot(x='Model', y='Test R2', data=results_df)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 11. Hyperparameter tuning of models

In [ ]:
# Parameter grids (kept small for speed)
param_grids = {
    'Ridge': {'model__alpha': [0.1, 1.0, 10.0]},
    'Lasso': {'model__alpha': [0.001, 0.01, 0.1]},
    'RandomForest': {
        'model__n_estimators': [100, 200],
        'model__max_depth': [None, 10]
    },
    'GradientBoosting': {
        'model__n_estimators': [100, 200],
        'model__learning_rate': [0.05, 0.1]
    }
}

In [ ]:
# Run GridSearchCV for selected models
tuned_models = {}
tuned_results = []
for name, grid in param_grids.items():
    base = models[name]
    gs = GridSearchCV(base, grid, cv=3, n_jobs=-1, scoring='r2')
    gs.fit(X_train, y_train)
    y_te = gs.predict(X_test)
    tuned_results.append({
        'Model': name + ' (tuned)',
+        
        'Test R2': r2_score(y_test, y_te),
+        
        'Test MAE': mean_absolute_error(y_test, y_te)
    })
    tuned_models[name] = gs.best_estimator_

In [ ]:
# Combine base and tuned results
tuned_df = pd.DataFrame(tuned_results)
all_results = pd.concat([results_df, tuned_df], ignore_index=True)
all_results.sort_values('Test R2', ascending=False)

## 12. Hybrid stacking model

In [ ]:
# StackingRegressor with linear + tree ensembles
stack_model = Pipeline([
    ('pre', preprocessor),
    ('fs', feature_selector),
    ('model', StackingRegressor(
        estimators=[
            ('ridge', Ridge()),
            ('rf', RandomForestRegressor(random_state=42)),
            ('gb', GradientBoostingRegressor(random_state=42))
        ],
        final_estimator=Ridge()
    ))
])

In [ ]:
# Train and evaluate stacking model
stack_model.fit(X_train, y_train)
y_te = stack_model.predict(X_test)
stack_row = {
    'Model': 'Stacking Hybrid',
    'Train R2': r2_score(y_train, stack_model.predict(X_train)),
    'Test R2': r2_score(y_test, y_te),
    'Test RMSE': np.sqrt(mean_squared_error(y_test, y_te)),
    'Test MAE': mean_absolute_error(y_test, y_te)
}
stack_row

In [ ]:
# Append hybrid to all results
all_results = pd.concat([all_results, pd.DataFrame([stack_row])], ignore_index=True)
all_results.sort_values('Test R2', ascending=False)

## 13. Final model selection and saving

In [ ]:
# Pick best model by Test R2
best_row = all_results.sort_values('Test R2', ascending=False).iloc[0]
best_name = best_row['Model']
print('Best model:', best_name)

In [ ]:
# Map best model name to fitted object
if best_name == 'Stacking Hybrid':
    final_model = stack_model
elif best_name.endswith(' (tuned)'):
    base_name = best_name.replace(' (tuned)', '')
    final_model = tuned_models[base_name]
else:
    final_model = fitted_models[best_name]
final_model

In [ ]:
# Save final best model as pkl
import os
os.makedirs('models', exist_ok=True)
dump(final_model, 'models/final_best_model.joblib')
'Saved to models/final_best_model.joblib'

## 14. Final model performance matrix

In [ ]:
# Full comparison of all models
all_results.sort_values('Test R2', ascending=False).reset_index(drop=True)